In [43]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)


In [44]:
import cbbd 

In [45]:
import time 
import os

from cbbd.rest import ApiException
from pprint import pprint

configuration = cbbd.Configuration(
    host = "https://api.collegebasketballdata.com"
)



In [46]:
api_key = os.getenv("CBBD_API_KEY")

if not api_key:
    for env_path in [".env", "../.env", "../../.env"]:
        if os.path.exists(env_path):
            with open(env_path, "r") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("CBBD_API_KEY") and "=" in line:
                        api_key = line.split("=", 1)[1].strip().strip("'").strip('"')
                        break
            if api_key:
                os.environ["CBBD_API_KEY"] = api_key
                break

if not api_key:
    raise KeyError("CBBD_API_KEY is not set. Define it in env or .env before creating configuration.")

configuration = cbbd.Configuration(
    host=configuration.host,
    access_token=api_key
)

In [47]:
with cbbd.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = cbbd.ConferencesApi(api_client)
    conference = 'BIG TEN' # str | Optional conference abbreviation filter (optional)

    try:
        api_response = api_instance.get_conference_history(conference=conference)
        print("The response of ConferencesApi->get_conference_history:\n")
        pprint(api_response)
    except ApiException as e:
        print("Exception when calling ConferencesApi->get_conference_history: %s\n" % e)


The response of ConferencesApi->get_conference_history:

[ConferenceHistory(id=10, source_id='7', name='Big Ten Conference', abbreviation='Big Ten', short_name='Big Ten', teams=[ConferenceHistoryTeamsInner(end_season=None, start_season=1899, mascot='Golden Gophers', school='Minnesota', source_id='135', id=173), ConferenceHistoryTeamsInner(end_season=None, start_season=1899, mascot='Badgers', school='Wisconsin', source_id='275', id=355), ConferenceHistoryTeamsInner(end_season=None, start_season=1900, mascot='Boilermakers', school='Purdue', source_id='2509', id=236), ConferenceHistoryTeamsInner(end_season=None, start_season=1901, mascot='Hoosiers', school='Indiana', source_id='84', id=121), ConferenceHistoryTeamsInner(end_season=None, start_season=1902, mascot='Hawkeyes', school='Iowa', source_id='2294', id=124), ConferenceHistoryTeamsInner(end_season=1946, start_season=1904, mascot=None, school='Chicago', source_id='UNKNOWN#Chicago', id=373), ConferenceHistoryTeamsInner(end_season=None,

In [48]:
with cbbd.ApiClient(configuration) as api_client:
    api_instance = cbbd.LinesApi(api_client)
    season = None  # int | Optional season filter (optional)
    team = "Michigan"  # str | Optional team name filter (optional)
    conference = None  # str | Optional conference abbreviation filter (optional)
    start_date_range = "2026-03-01T00:00:00Z"  # datetime | Optional start timestamp in ISO 8601 format (optional)
    end_date_range = None  # datetime | Optional end timestamp in ISO 8601 format (optional)

    try:
        api_response = api_instance.get_lines(
            season=season,
            team=team,
            conference=conference,
            start_date_range=start_date_range,
            end_date_range=end_date_range,
        )
        print("The response of LinesApi->get_lines:\n")
        pprint(api_response)

        normalized_rows = [
            row for row in (_normalize_line_record(line) for line in api_response)
            if isinstance(row, dict)
        ]

        if not normalized_rows:
            print("Lines API returned 0 records after normalization.")
            lines_df = pd.DataFrame()
        else:
            lines_df = pd.DataFrame(normalized_rows)
            print(f"Lines dataframe shape: {lines_df.shape}")
            print(lines_df.head())
    except Exception as e:
        print("Exception when calling LinesApi->get_lines: %s\n" % e)


The response of LinesApi->get_lines:

[GameLines(game_id=215200, season=2026, season_type=<SeasonType.REGULAR: 'regular'>, start_date=datetime.datetime(2026, 3, 6, 1, 0, tzinfo=datetime.timezone.utc), home_team_id=124, home_team='Iowa', home_conference='Big Ten', home_score=68, away_team_id=170, away_team='Michigan', away_conference='Big Ten', away_score=71, lines=[GameLineInfo(provider='Draft Kings', spread=8.5, over_under=146.5, home_moneyline=380, away_moneyline=-500, spread_open=9.5, over_under_open=145.5), GameLineInfo(provider='Bovada', spread=9, over_under=147, home_moneyline=320, away_moneyline=-430, spread_open=8.5, over_under_open=145.5)]),
 GameLines(game_id=215240, season=2026, season_type=<SeasonType.REGULAR: 'regular'>, start_date=datetime.datetime(2026, 3, 8, 20, 30, tzinfo=datetime.timezone.utc), home_team_id=170, home_team='Michigan', home_conference='Big Ten', home_score=90, away_team_id=169, away_team='Michigan State', away_conference='Big Ten', away_score=80, lines=

In [49]:
lines_df

,gameId,season,seasonType,startDate,homeTeamId,homeTeam,homeConference,homeScore,awayTeamId,awayTeam,awayConference,awayScore,lines
0,215200,2026,SeasonType.REGULAR,2026-03-06 01:00:00+00:00,124,Iowa,Big Ten,68,170,Michigan,Big Ten,71,"[{'provider': 'Draft Kings', 'spread': 8.5, 'o..."
1,215240,2026,SeasonType.REGULAR,2026-03-08 20:30:00+00:00,170,Michigan,Big Ten,90,169,Michigan State,Big Ten,80,"[{'provider': 'Draft Kings', 'spread': -10.5, ..."
2,372394,2026,SeasonType.REGULAR,2026-03-13 16:00:00+00:00,170,Michigan,Big Ten,71,216,Ohio State,Big Ten,67,"[{'provider': 'Bovada', 'spread': -12.5, 'over..."
3,372427,2026,SeasonType.REGULAR,2026-03-14 17:00:00+00:00,170,Michigan,Big Ten,68,355,Wisconsin,Big Ten,65,"[{'provider': 'Draft Kings', 'spread': -12.5, ..."
4,372449,2026,SeasonType.REGULAR,2026-03-15 19:30:00+00:00,170,Michigan,Big Ten,72,236,Purdue,Big Ten,80,"[{'provider': 'Bovada', 'spread': -5, 'overUnd..."
5,372514,2026,SeasonType.POSTSEASON,2026-03-19 04:00:00+00:00,170,Michigan,Big Ten,0,1217,TBD,NaN,0,[]


In [50]:
def _normalize_line_record(line_obj):
    """Convert API GameLines objects into plain dicts with nested lines normalized."""
    if hasattr(line_obj, "to_dict"):
        record = line_obj.to_dict()
    elif isinstance(line_obj, dict):
        record = dict(line_obj)
    else:
        return line_obj

    if isinstance(record, dict):
        raw_lines = record.get("lines")
        if raw_lines is not None:
            normalized_lines = []
            for entry in raw_lines:
                if hasattr(entry, "to_dict"):
                    normalized_lines.append(entry.to_dict())
                elif isinstance(entry, dict):
                    normalized_lines.append(entry)
                else:
                    normalized_lines.append(entry)
            record["lines"] = normalized_lines

    return record


In [51]:
def fetch_lines(team_names, start_date, end_date=None, season=None, conference=None):
    """Return game-level and line-level DataFrames for the requested teams."""
    if isinstance(team_names, str):
        teams = [team_names]
    else:
        teams = list(team_names)

    if not teams:
        raise ValueError("team_names must contain at least one team.")

    all_games = []
    all_line_items = []

    with cbbd.ApiClient(configuration) as api_client:
        api_instance = cbbd.LinesApi(api_client)
        for team in teams:
            api_response = api_instance.get_lines(
                season=season,
                team=team,
                conference=conference,
                start_date_range=start_date,
                end_date_range=end_date,
            )

            response_rows = [
                row for row in (_normalize_line_record(line) for line in api_response)
                if isinstance(row, dict)
            ]
            if not response_rows:
                continue

            games_df = pd.DataFrame(response_rows)
            games_df["team_query"] = team
            all_games.append(games_df)

            line_records = []
            for record in response_rows:
                parent_values = {k: v for k, v in record.items() if k != "lines"}
                parent_values["team_query"] = team
                for line_entry in record.get("lines") or []:
                    line_row = parent_values.copy()
                    if isinstance(line_entry, dict):
                        line_row.update({f"line_{k}": v for k, v in line_entry.items()})
                    else:
                        line_row["line_entry"] = line_entry
                    line_records.append(line_row)

            if line_records:
                all_line_items.append(pd.DataFrame(line_records))

    games_out = pd.concat(all_games, ignore_index=True) if all_games else pd.DataFrame()
    line_items_out = pd.concat(all_line_items, ignore_index=True) if all_line_items else pd.DataFrame()

    return games_out, line_items_out


In [52]:
games_df, line_items_df = fetch_lines(
    team_names=["Michigan", "Duke"],
    start_date="2026-03-01T00:00:00Z"
)


In [53]:
lines_df

,gameId,season,seasonType,startDate,homeTeamId,homeTeam,homeConference,homeScore,awayTeamId,awayTeam,awayConference,awayScore,lines
0,215200,2026,SeasonType.REGULAR,2026-03-06 01:00:00+00:00,124,Iowa,Big Ten,68,170,Michigan,Big Ten,71,"[{'provider': 'Draft Kings', 'spread': 8.5, 'o..."
1,215240,2026,SeasonType.REGULAR,2026-03-08 20:30:00+00:00,170,Michigan,Big Ten,90,169,Michigan State,Big Ten,80,"[{'provider': 'Draft Kings', 'spread': -10.5, ..."
2,372394,2026,SeasonType.REGULAR,2026-03-13 16:00:00+00:00,170,Michigan,Big Ten,71,216,Ohio State,Big Ten,67,"[{'provider': 'Bovada', 'spread': -12.5, 'over..."
3,372427,2026,SeasonType.REGULAR,2026-03-14 17:00:00+00:00,170,Michigan,Big Ten,68,355,Wisconsin,Big Ten,65,"[{'provider': 'Draft Kings', 'spread': -12.5, ..."
4,372449,2026,SeasonType.REGULAR,2026-03-15 19:30:00+00:00,170,Michigan,Big Ten,72,236,Purdue,Big Ten,80,"[{'provider': 'Bovada', 'spread': -5, 'overUnd..."
5,372514,2026,SeasonType.POSTSEASON,2026-03-19 04:00:00+00:00,170,Michigan,Big Ten,0,1217,TBD,NaN,0,[]


In [54]:
lines_df.lines[0]

[{'provider': 'Draft Kings',
  'spread': 8.5,
  'overUnder': 146.5,
  'homeMoneyline': 380,
  'awayMoneyline': -500,
  'spreadOpen': 9.5,
  'overUnderOpen': 145.5},
 {'provider': 'Bovada',
  'spread': 9,
  'overUnder': 147,
  'homeMoneyline': 320,
  'awayMoneyline': -430,
  'spreadOpen': 8.5,
  'overUnderOpen': 145.5}]